# 12 -- Album Art Generation (Stable Diffusion, local/free tools only)

Generates one album-art image per song from the deterministic, feature-grounded prompts `scripts/export_album_art_prompts.py` already produced (`album_art/prompts.json`) -- **this notebook does not build prompts, it only turns already-built prompts into images.** See that script and `sonic_explorer/analysis/album_art_prompt.py` for how each prompt's phrases trace back to real per-song audio features (brightness, energy, key, tempo, AST sound tags).

**No paid API** -- runs entirely on Colab's free-tier GPU (T4) via Hugging Face `diffusers`, using **SD-Turbo** (`stabilityai/sd-turbo`), a distilled Stable Diffusion checkpoint built for 1-4 step inference -- the right tradeoff here: this is a one-time batch job over ~200+ songs, not a quality-critical single image, so generation speed matters far more than squeezing out the last bit of fidelity a 50-step full SDXL run would give.

**Deliberately stops here.** This notebook produces PNGs in `/content/album_art_output/` and a downloadable zip -- it does NOT wire anything into the deployed app (that's step 5, held off until real generated images exist to look at and decide on).

## 1. Install `diffusers` + confirm GPU

Runtime -> Change runtime type -> GPU (T4 is enough for SD-Turbo). This notebook does **not** need the `sonic_explorer` package at all -- it only reads a plain JSON file of already-built prompts, no DB/audio access happens here.

In [ ]:
!pip install -q diffusers accelerate transformers

import torch

print('GPU available:', torch.cuda.is_available(), '--',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only (this will be slow -- switch runtime type)')

## 2. Upload `prompts.json`

From your local `album_art/prompts.json` (produced by `python scripts/export_album_art_prompts.py`, run locally first -- see that script's own docstring). A small JSON file (~a few hundred KB for ~200 songs), so a direct upload is simpler here than mounting Drive.

In [ ]:
from google.colab import files

uploaded = files.upload()  # pick album_art/prompts.json from your local machine
prompts_path = next(iter(uploaded))
print('Uploaded:', prompts_path)

## 3. Load the prompts

In [ ]:
import json

with open(prompts_path) as f:
    rows = json.load(f)

print(f'{len(rows)} prompts loaded')
print()
print('Example:')
print(f"  song_id={rows[0]['song_id']}  {rows[0]['title']!r} by {rows[0]['artist']}")
print(f"  prompt: {rows[0]['prompt']}")

## 4. Load SD-Turbo

`num_inference_steps=1` and `guidance_scale=0.0` are SD-Turbo's own intended settings (it's distilled specifically for single-step, no-CFG inference -- passing a real guidance_scale here would fight how the checkpoint was trained, not improve quality). `fp16` halves memory/time on a T4 with no visible quality loss for this use case.

In [ ]:
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained(
    'stabilityai/sd-turbo', torch_dtype=torch.float16, variant='fp16',
)
pipe = pipe.to('cuda' if torch.cuda.is_available() else 'cpu')
print('SD-Turbo loaded.')

## 5. Smoke test -- generate a few images and look at them before the full run

Same discipline as this project's other batch notebooks (see `11_sound_tags_facet.ipynb`): check the real output looks right, and measure your actual per-image time on this runtime before committing to the full batch -- the time estimate below is what SD-Turbo's own release reports on a T4-class GPU, not something measured in the environment this notebook was written in (no GPU was available there).

In [ ]:
import time
from IPython.display import display

sample_rows = rows[:3]
times = []
for row in sample_rows:
    t0 = time.time()
    image = pipe(
        prompt=row['prompt'], num_inference_steps=1, guidance_scale=0.0,
    ).images[0]
    elapsed = time.time() - t0
    times.append(elapsed)
    print(f"song_id={row['song_id']}  {row['title']!r}  ({elapsed:.2f}s)")
    print(f"  {row['prompt']}")
    display(image)

mean_rate = sum(times) / len(times)
print(f'\nMean time per image on this runtime: {mean_rate:.2f}s')
print(f'Estimated full run ({len(rows)} songs): ~{mean_rate * len(rows) / 60:.1f} minutes')

## 6. Full batch run

Saved as `{song_id}.png` -- matches the key the exported prompts (and, later, the app's own wiring step) already use, no separate id-mapping file needed. Checkpointed print progress only (no resume-skip logic) -- at SD-Turbo speeds this whole batch should be well under the length of a typical Colab session, so the added complexity of a resumable checkpoint (like the sound-tags notebook needs for its multi-hour run) isn't worth it here; if a run does get interrupted, just re-run this cell -- it always regenerates every image from the same deterministic prompts, so nothing is lost by starting over.

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path('/content/album_art_output')
OUTPUT_DIR.mkdir(exist_ok=True)

_start = time.time()
for i, row in enumerate(rows):
    image = pipe(
        prompt=row['prompt'], num_inference_steps=1, guidance_scale=0.0,
    ).images[0]
    image.save(OUTPUT_DIR / f"{row['song_id']}.png")

    if (i + 1) % 25 == 0:
        elapsed_min = (time.time() - _start) / 60
        print(f'  ...{i + 1}/{len(rows)} images generated ({elapsed_min:.1f} min elapsed)')

print(f'\nDone. {len(rows)} images written to {OUTPUT_DIR}.')

## 7. Zip and download

In [ ]:
import shutil

zip_path = shutil.make_archive('/content/album_art_output', 'zip', OUTPUT_DIR)
print('Zipped to', zip_path)

from google.colab import files
files.download(zip_path)

## Done -- and what's still manual after this

You now have a local zip of `{song_id}.png` files, one per song in the exported prompt set. **Deliberately stopping here** -- per explicit scope, wiring these into the Streamlit app (a new `album_art_path_for(song)`-style resolver, a fallback for songs without generated art, and surfacing the image somewhere in Explore/Song X-Ray) is step 5, held off until there are real generated images to look at first and decide how they should actually be used in the UI -- not before.

Before that step: unzip these into the repo (e.g. `deploy_data/album_art/` or wherever step 5 ends up deciding on) and take a look at whether SD-Turbo's output at this prompt style/length is actually good enough, or whether the prompt templates in `sonic_explorer/analysis/album_art_prompt.py` need another pass first.